# Assignment: Semantic Meaning

This notebook analyzes **5 word pairs** and explains their **semantic similarity** using Python.

## Objective
- Compute semantic similarity scores for 5 word pairs.
- Interpret each pair's relationship.
- Provide final work description and learning outcomes.

## 1. Import Libraries and Configure Notebook

We import required libraries and prepare the notebook environment.

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import wordnet as wn

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.precision', 3)

print('Libraries imported successfully.')

## 2. Define the 5 Word Pairs

Create exactly five pairs with mixed similarity levels (high, medium, low).

In [ ]:
word_pairs = [
    ('happy', 'joyful'),    # high
    ('car', 'vehicle'),     # high-medium (hypernym relation)
    ('teacher', 'school'),  # medium (contextually related)
    ('book', 'library'),    # medium (contextually related)
    ('banana', 'justice')   # low (mostly unrelated)
]

pairs_df = pd.DataFrame(word_pairs, columns=['word_1', 'word_2'])
pairs_df

## 3. Load a Semantic Similarity Model

Use NLTK WordNet as a lexical resource and validate access before scoring.

In [ ]:
# Download WordNet resources if they are missing.
try:
    _ = wn.synsets('test')
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4')


def best_similarity_score(word1: str, word2: str) -> float:
    """Return the highest Wu-Palmer similarity between synset pairs."""
    w1 = word1.lower().strip()
    w2 = word2.lower().strip()

    synsets1 = wn.synsets(w1)
    synsets2 = wn.synsets(w2)

    if not synsets1 or not synsets2:
        return 0.0

    best = 0.0
    for s1 in synsets1:
        for s2 in synsets2:
            score = s1.wup_similarity(s2)
            if score is not None:
                best = max(best, float(score))
    return round(best, 3)

print('WordNet is ready for semantic similarity analysis.')

## 4. Compute Similarity Scores for Each Pair

Calculate numeric similarity scores for all 5 pairs and present them clearly.

In [ ]:
results_df = pairs_df.copy()
results_df['similarity_score'] = results_df.apply(
    lambda row: best_similarity_score(row['word_1'], row['word_2']),
    axis=1
)

results_df = results_df.sort_values(by='similarity_score', ascending=False).reset_index(drop=True)
results_df

## 5. Generate Pair-wise Explanations of Semantic Similarity

Interpret each score and describe the semantic relationship for every pair.

In [ ]:
def relation_label(score: float) -> str:
    if score >= 0.85:
        return 'Synonym-like / Very close meaning'
    if score >= 0.60:
        return 'Strongly related meaning'
    if score >= 0.35:
        return 'Weak to moderate relation'
    return 'Unrelated or very weak relation'


def explanation_text(word1: str, word2: str, score: float) -> str:
    label = relation_label(score)
    if label == 'Synonym-like / Very close meaning':
        return f"'{word1}' and '{word2}' are nearly interchangeable in many contexts."
    if label == 'Strongly related meaning':
        return f"'{word1}' and '{word2}' are closely connected, but may differ in specificity or usage."
    if label == 'Weak to moderate relation':
        return f"'{word1}' and '{word2}' share some context, but they are not direct synonyms."
    return f"'{word1}' and '{word2}' usually appear in different semantic contexts."

results_df['relationship_type'] = results_df['similarity_score'].apply(relation_label)
results_df['explanation'] = results_df.apply(
    lambda row: explanation_text(row['word_1'], row['word_2'], row['similarity_score']),
    axis=1
)

final_results_df = results_df[['word_1', 'word_2', 'similarity_score', 'relationship_type', 'explanation']]
final_results_df